# FoodShock — Cross-Country ASEAN Food Price Forecasting & Early Shock Detection System

**Machine Learning Solution Challenge (MLSC) – Cyan Academic Competition 2026**  
**Organizer:** Enterprise Data Management (EDM) Laboratory  
**Target SDG:** SDG 2 — Zero Hunger  
**Solution Name:** FoodShock  

---

## 1. Perumusan Permasalahan & Keterkaitan SDG 2 (15%)

Ketahanan pangan ASEAN menghadapi tekanan struktural dan siklikal. Fluktuasi harga pangan pokok (beras, daging, minyak goreng) langsung memengaruhi daya beli rumah tangga miskin dan stabilitas sosial. Data WFP/HDX 2015–2025 menunjukkan heterogenitas harga antar negara dan pasar yang tinggi.

**Urgensi empiris & keterkaitan SDG 2:**
- Harga beras dan daging adalah komponen dominan indeks harga pangan rumah tangga di IDN, PHL, LAO, KHM, MMR, THA, TLS.
- Shock harga t+1 yang tidak terdeteksi dini mengurangi efektivitas intervensi (operasi pasar, subsidi tepat sasaran).
- SDG 2 (Zero Hunger) menargetkan stabilitas pasokan dan keterjangkauan harga pangan pokok. Sistem early-warning berbasis ML panel antar-negara memberikan instrumen konkret bagi AFSIS dan kementerian nasional.


## 2. Metodologi & Aturan Teknis

- Audit seri ≥ 60 bulan; 7 negara (IDN, KHM, LAO, PHL, THA, TLS, MMR).
- 5 kelompok komoditas baku didukung ≥ 3 negara: Rice, Chicken Meat, Beef Meat, Pork Meat, Cooking Oil.
- Harga USD, unit distandardisasi (KG / L).
- **Pemisahan eksplisit** `pricetype` (Retail vs Wholesale) dan `priceflag`.
- Calendar-safe re-index **tanpa forward-fill** harga mentah → zero leakage.
- **Target residual**: Δy_t+1 = y_t+1 − y_t (menghilangkan bias level antar komoditas), lalu rekonstruksi ŷ_t+1 = y_t + Δŷ_t+1.
- Walk-forward / time-based hold-out (18 bulan terakhir).
- Metrik: MAE, RMSE, sMAPE, R² + klasifikasi shock (Precision, Recall, F1, ROC-AUC).


## 3. Tahap 1 — Audit Cakupan

### Country Coverage (series ≥ 60 months)
```
  countryiso3  n_series  n_commodity_groups  avg_months  max_months    min_date    max_date
0         PHL       305                   4   68.826230         116  2015-01-15  2025-12-15
1         IDN       100                   4   60.080000          62  2015-01-15  2024-05-15
2         LAO        82                   5  111.170732         127  2015-01-15  2025-12-15
3         KHM        69                   3   69.260870          88  2015-01-15  2025-12-15
4         MMR        17                   2   77.352941         105  2015-01-15  2025-12-15
5         TLS         7                   3   71.142857          94  2015-01-15  2025-12-15
6         THA         2                   2   63.000000          63  2015-01-15  2020-03-15
```

### Commodity Group Coverage
```
  commodity_group  n_series  n_countries                    countries  avg_months  avg_price_usd
0            Rice       204            7  IDN,KHM,LAO,MMR,PHL,THA,TLS   71.112745       0.766926
1     Cooking Oil        53            5          IDN,KHM,LAO,MMR,TLS   78.150943       1.419214
2    Chicken Meat        98            5          IDN,LAO,PHL,THA,TLS   74.459184       3.457132
3       Beef Meat       113            3                  IDN,LAO,PHL   73.495575       6.692020
4       Pork Meat       114            3                  KHM,LAO,PHL   75.289474       4.965520
```

**Catatan penting tentang IDN & THA di training set saja:**  
IDN data berakhir Mei 2024 dan THA berakhir Maret 2020. Karena test set adalah *out-of-time* 18 bulan terakhir (Juni 2024 – November 2025), kedua negara secara alami hanya muncul di training set. Ini sesuai prinsip time-based validation dan menghindari leakage dari masa depan.


## 4. Tahap 2 — Feature Engineering (Zero Leakage)

Fitur: lag_1/2/3/6/12, rolling mean/std/min/max (dari shift(1)), price_to_mean6, cv_6, pct_change_1m/3m/12m, month_sin/cos, countryiso3, commodity_group, pricetype, priceflag.

Target residual: residual_target = target_price_usd_t1 − lag_1.

Model-ready shape: 28.172 baris | 2016-02 → 2025-11 | 7 negara | 5 commodity groups.


### Tren Harga Rata-rata
![EDA Trends](figures/eda_trends.png)


## 5. Tahap 3 — Residual Forecasting & Model Comparison

**Mengapa residual?** Target harga absolut mencampur skala Rice (~USD 0.77) dengan Beef (~USD 6.7). Tree models mengalami bias level. Dengan memprediksi residual lalu merekonstruksi, model fokus pada perubahan.

### Overall Comparison (Absolute Price after Residual Reconstruction)
```
                  model      MAE     RMSE    sMAPE       R2
    Persistence (lag_1) 0.147863 0.247235 3.805180 0.988737
LightGBM_Tuned_Residual 0.150875 0.244491 3.907877 0.988986
           RandomForest 0.158076 0.249123 4.080658 0.988564
                XGBoost 0.161470 0.256107 4.139442 0.987914
   HistGradientBoosting 0.162737 0.255599 4.197799 0.987962
               CatBoost 0.163909 0.256224 4.217679 0.987903
               LightGBM 0.164053 0.257979 4.199611 0.987737
Seasonal Naive (lag_12) 0.314154 0.475581 9.146928 0.958325
```

### Perbandingan Sebelum vs Sesudah Residual

| Approach | Best ML MAE | Persistence MAE | Catatan |
|----------|-------------|-----------------|---------|
| Absolute price (sebelum) | 0.1626 | 0.1479 | ML kalah jauh |
| **Residual reconstruction (sesudah)** | **0.1509** | 0.1479 | ML mendekati; **RMSE & R² unggul** |

- RMSE Tuned Residual **0.2445** < Persistence **0.2472**
- R² Tuned Residual **0.9890** > Persistence **0.9887**


### Visualisasi
![Model Comparison](figures/model_comparison.png)
![Actual vs Predicted](figures/actual_vs_predicted.png)


### Evaluasi Stratified

**By Commodity Group**
```
commodity_group      MAE     RMSE    sMAPE       R2      n
      Beef Meat 0.197738 0.266867 3.025000 0.915222 1665.0
   Chicken Meat 0.174186 0.295936 4.586072 0.597699 1348.0
    Cooking Oil 0.060094 0.135911 3.600830 0.742284  332.0
      Pork Meat 0.235267 0.313918 4.310230 0.895709 1728.0
           Rice 0.033000 0.047795 3.888222 0.885678 1910.0
```

**By Country**
```
countryiso3      MAE     RMSE     sMAPE       R2      n
        KHM 0.080932 0.125991  4.849253 0.992468  352.0
        LAO 0.113025 0.271707  3.234236 0.974798 1222.0
        MMR 0.071285 0.125861  4.726875 0.831398    6.0
        PHL 0.163557 0.241350  3.923063 0.989729 5359.0
        TLS 0.227829 0.461154 13.124463 0.868095   44.0
```

**By Price Type (Retail vs Wholesale)**
```
pricetype      MAE     RMSE    sMAPE       R2      n
   Retail 0.152683 0.246763 3.875991 0.988724 6812.0
Wholesale 0.078866 0.123696 5.178098 0.992261  171.0
```

Retail (n≈6812) MAE≈0.153; Wholesale (n≈171) MAE≈0.079. Evaluasi terpisah mencegah bias margin grosir vs eceran.


## 6. Food Price Shock Early-Warning Classification (Inovasi SDG 2)

Definisi shock: |%Δ_t+1| > 5% **atau** residual > 1.5× std historis training.

```
 Precision   Recall       F1  ROC_AUC  Support_Shock  Support_Total  Shock_Rate
  0.254237 0.013286 0.025253 0.554395           1129           6983    0.161678
```

![Shock Residual Distribution](figures/shock_residual_dist.png)

Shock adalah peristiwa langka (~16%). Model residual memberikan sinyal (ROC-AUC ≈ 0.55). Precision/Recall rendah mencerminkan kesulitan intrinsik prediksi lonjakan satu langkah ke depan; skor soft dapat dikalibrasi lebih lanjut.


## 7. SHAP Interpretability

![SHAP Summary](figures/shap_summary.png)

Setelah residual modeling, kontribusi fitur momentum, volatilitas, dan musiman menjadi lebih menonjol di samping lag_1. Encoding negara menangkap heterogenitas struktural.


## 8. Kesimpulan & Rekomendasi Kebijakan

FoodShock telah:
1. Mengatasi bias level dengan residual forecasting → ML mendekati Persistence pada MAE dan **mengungguli** pada RMSE & R².
2. Menyediakan evaluasi terpisah Retail vs Wholesale (`model_eval_by_pricetype.csv`).
3. Menghasilkan skor early-warning shock t+1 dengan metrik klasifikasi.
4. Memenuhi seluruh syarat teknis kompetisi (audit ≥60 bulan, zero leakage, time-based validation, multi-model + Optuna, SHAP).

**Rekomendasi kebijakan (SDG 2):**
- Integrasikan skor FoodShock ke dashboard AFSIS bulanan.
- Prioritaskan operasi pasar / pelepasan stok pada komoditas & pasar yang mendapat alert t+1 (terutama beras & minyak goreng).
- Gabungkan prediksi harga dengan data kerentanan rumah tangga untuk targeting subsidi yang lebih akurat.
